In [1]:
import torch

In [2]:
m = 7
l = 9
vocab_size = 9
target = [0, 1, 2, 3, 4, 5, 8]
assert len(target) == m
target = torch.tensor(target)
p = 3
p_idx = -1
mstar = m + p
lstar = l + p
target = torch.cat([target, torch.tensor([p_idx] * p)])

In [3]:
target

tensor([ 0,  1,  2,  3,  4,  5,  8, -1, -1, -1])

In [4]:
transition_matrix = torch.zeros((lstar, lstar))

In [5]:
transition_matrix.shape

torch.Size([12, 12])

In [6]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        1.0 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),

    ## these transitions are dummy transitions for the padding
    ## for any (x,y), we must have 9 <= x and 10 <= y
    ## and y >= x + 1
    (
        (9, 10), #row, col
        0.5 #prob
    ),
    (
        (9, 11), #row, col
        0.5 #prob
    ),
    (
        (10, 12), #row, col
        1.0 #prob
    ),
    (
        (11, 12), #row, col
        1.0 #prob
    )
]

In [7]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [8]:
transition_matrix

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

In [9]:
transition_matrix[m]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])

In [10]:
token_probs = torch.zeros((lstar, vocab_size))

In [11]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.9 #prob
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.3 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    ),
    ## padding token emissions
    ## for any (x, y), we must have x >= 10 and 1 <= y <= 9
    (
        (10, 1), #row, col
        0.1 #prob
    ),
    (
        (10, 3), #row, col
        0.9 #prob
    ),
    (
        (11, 4), #row, col
        1.0 #prob
    ),
    (
        (12, 5), #row, col
        1.0 #prob
    )
]

In [12]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [13]:
dp = torch.zeros((mstar, lstar))

In [14]:
dp.shape

torch.Size([10, 12])

In [15]:
dp[0, 0] = 0.9

In [16]:
# turn transition matrix, token probs, and dp into log space
transition_matrix = torch.log(transition_matrix)
token_probs = torch.log(token_probs)
dp = torch.log(dp)

In [17]:
# think of the above as the result of what you get from performing log_softmax
# on the output logits of the model

# first mask the lower triangle of the transition matrix (including the diagonal)
# this prevents cycles from being formed in the graph defined by the transition matrix
acyclic_mask = torch.triu(torch.ones_like(transition_matrix), diagonal=1)

In [18]:
# all positions where acyclic_mask == 0 need to be set to -inf
transition_matrix = transition_matrix.masked_fill(acyclic_mask == 0, float('-inf'))

In [19]:
# now need to figure out which tokens in the target are

In [20]:
dp.shape, transition_matrix.shape

(torch.Size([10, 12]), torch.Size([12, 12]))

In [21]:
for i in range(1, m):
    dp[i, :] = token_probs[:, target[i]] + (torch.logsumexp(dp[i - 1, :].unsqueeze(0).T + transition_matrix, dim=0))

In [22]:
token_probs[:, target[0]].shape

torch.Size([12])

In [23]:
# handling padding
padding_token_probs = torch.zeros_like(token_probs[:, target[0]])
padding_transition_matrix = torch.zeros_like(transition_matrix)

In [24]:
padding_transition_matrix = padding_transition_matrix - float('inf')

In [25]:
padding_transition_matrix[8][9] = 0
padding_transition_matrix[9][10] = 0
padding_transition_matrix[10][11] = 0

In [26]:
dp[7 - 1, :]

tensor([   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -6.7050,
        -3.6602,    -inf,    -inf,    -inf])

In [27]:
for i in range(m, mstar):
    dp[i, :] = padding_token_probs + (torch.logsumexp(dp[i - 1, :].unsqueeze(0).T + padding_transition_matrix, dim=0))

In [28]:
print(dp.round(decimals=3))

tensor([[ -0.1050,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,  -1.6660,  -2.0710,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,  -1.8890,  -4.3740,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,  -1.9950,  -6.6770,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,  -2.1000,     -inf,  -9.6720,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,     -inf,  -3.3040,  -5.0960,
         -11.2820,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
          -6.7050,  -3.6600,     -inf,     -inf,     -inf],
        [    -inf,     -inf

In [29]:
# doing it by hand by enumerating all possible paths
p1m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.6, 0.7])
p1t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

# p2m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.1, 0.2])
# p2t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

p3m = torch.tensor([0.9, 0.2, 0.1, 0.1, 0.1, 0.2, 0.7])
p3t = torch.tensor([0.7, 1, 1, 0.5, 1, 1])

In [30]:
p1 = p1m.prod() * p1t.prod()
# p2 = p2m.prod() * p2t.prod()
p3 = p3m.prod() * p3t.prod()

In [31]:
# acc = p1 + p2 + p3
acc = p1 + p3

In [32]:
acc

tensor(0.0257)

In [33]:
dp

tensor([[ -0.1054,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,  -1.6660,  -2.0715,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,  -1.8892,  -4.3741,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,  -1.9945,  -6.6766,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,  -2.0999,     -inf,  -9.6724,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,     -inf,  -3.3038,  -5.0956,
         -11.2818,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
          -6.7050,  -3.6602,     -inf,     -inf,     -inf],
        [    -inf,     -inf

In [34]:
# check difference between dp[m-1, l-1] and acc
dp = torch.exp(dp)
dp[-1, -1] - acc

tensor(-1.8626e-09)

This padding method seems to work, now the only problem is how to create the appropriate emission and transition matrices (currently doing it by hand)